In [4]:
### 验证图片的像素值
from PIL import Image
import numpy as np
from pathlib import Path

mask_dir = Path('./pv08_prepared/val/masks/')
paths = sorted(mask_dir.glob('*'))[:5]
for p in paths:
    arr = np.array(Image.open(p))
    u = np.unique(arr)
    print(p.name, 'min', arr.min(), 'max', arr.max(), 'unique(sample)', u[:10], 'len', len(u))

PV08_313806_1198546_label.bmp min 0 max 255 unique(sample) [  0 255] len 2
PV08_313942_1194411_label.bmp min 0 max 255 unique(sample) [  0 255] len 2
PV08_314619_1199320_label.bmp min 0 max 255 unique(sample) [  0 255] len 2
PV08_314693_1199320_label.bmp min 0 max 255 unique(sample) [  0 255] len 2
PV08_314910_1196475_label.bmp min 0 max 255 unique(sample) [  0 255] len 2


In [3]:
from PIL import Image
import numpy as np
from pathlib import Path

mask_dirs = [
    Path('./pv08_prepared/train/masks/'),
    Path('./pv08_prepared/val/masks/'),
]

for mask_dir in mask_dirs:
    for p in mask_dir.glob('*'):
        arr = np.array(Image.open(p).convert('L'))
        # 只要 >0 就当作前景
        bin_ = (arr > 0).astype(np.uint8)*255
        Image.fromarray(bin_).save(p)
print('done')

done


In [5]:
import yaml
import torch
from torch.utils.data import DataLoader
import datasets

cfg = yaml.safe_load(open('configs/cod-sam-vit-b.yaml'))
spec = cfg['val_dataset']
ds = datasets.make(spec['dataset'])
ds = datasets.make(spec['wrapper'], args={'dataset': ds})
loader = DataLoader(ds, batch_size=1, num_workers=0)
batch = next(iter(loader))

gt = batch['gt']
print('gt min/max/mean:', gt.min().item(), gt.max().item(), gt.mean().item())
print('gt >0.5 ratio:', (gt > 0.5).float().mean().item())

/mnt/data/conda_envs/yanyi2025/rsam-seg/lib/python3.10/site-packages/torchvision/transforms/transforms.py:329: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  warnings.warn(


gt min/max/mean: 0.0 1.0 0.055603981018066406
gt >0.5 ratio: 0.055603981018066406


In [9]:
import yaml, torch
from torch.utils.data import DataLoader
import datasets, models

cfg = yaml.safe_load(open('configs/cod-sam-vit-b.yaml'))
spec = cfg['val_dataset']
ds = datasets.make(spec['dataset'])
ds = datasets.make(spec['wrapper'], args={'dataset': ds})
loader = DataLoader(ds, batch_size=1, num_workers=0)
batch = next(iter(loader))

model = models.make(cfg['model']).cuda()
ckpt = torch.load('save/exp_vitb/model_epoch_last.pth', map_location='cuda')
model.load_state_dict(ckpt, strict=False)
model.eval()

with torch.no_grad():
    pred = torch.sigmoid(model.infer(batch['inp'].cuda()))

thr=0.1
print('pred min/max/mean:', pred.min().item(), pred.max().item(), pred.mean().item())
print('pred >0.1 ratio:', (pred > thr).float().mean().item())

/mnt/data/conda_envs/yanyi2025/rsam-seg/lib/python3.10/site-packages/torchvision/transforms/transforms.py:329: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  warnings.warn(


pred min/max/mean: 0.14934378862380981 0.19150438904762268 0.17120587825775146
pred >0.1 ratio: 1.0
